In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, log_loss, balanced_accuracy_score, f1_score
from os import listdir
import cv2
import os

### Global variables

In [3]:
RANDOM_SEED = 0
IMG_SIZE = (64, 64)

## Data Preprocessing

### Extracting features from the images

In [4]:
FOLDER = "olivetti_faces"
SEPARATOR = '/'

X = []
y = []

for d in listdir(FOLDER):
    path = FOLDER+SEPARATOR+d+SEPARATOR
    if not os.path.isdir(path):
        continue
    for f in listdir(path):
        if ".jpg" in f:
            x = cv2.imread(path+f, cv2.IMREAD_GRAYSCALE)
            x = x.reshape(64*64,) # reshape to flatten the 64x64 pixel-matrix to a 1d array
            X.append(x)
            y.append(d)

X = np.array(X)
y = np.array(y)

### Splitting and normalizing the dataset

In [5]:
# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=RANDOM_SEED, stratify=y)

# Normalize
#X_max = X_train.max()

#X_train = X_train/X_max
#X_test = X_test/X_max

ss = StandardScaler()
X_train = ss.fit_transform(X_train)
X_test = ss.transform(X_test)

X_train.max()
X_train.min()

np.float64(-5.59169360563996)

#### Quick test to check if all the classes are both in the test and train sets

In [6]:
(1-np.isin(y_test, y_train)).sum()

np.int64(0)

In [7]:
Ks = [1, 2, 3, 4, 5, 10, 12, 15, 20, 30, 50, 100]

rows = []

for K in Ks:
    knn = KNeighborsClassifier(n_neighbors=K, weights="distance")
    knn.fit(X_train, y_train)

    # predictions
    y_pred_train = knn.predict(X_train)
    y_proba_train = knn.predict_proba(X_train)

    y_pred_test = knn.predict(X_test)
    y_proba_test = knn.predict_proba(X_test)

    labels = knn.classes_

    rows.append({
        "K": K,

        # TRAIN metrics
        "train_acc": accuracy_score(y_train, y_pred_train),
        "train_bal_acc": balanced_accuracy_score(y_train, y_pred_train),
        "train_f1_macro": f1_score(y_train, y_pred_train, average="macro"),
        "train_logloss": log_loss(y_train, y_proba_train, labels=labels),

        # TEST metrics
        "test_acc": accuracy_score(y_test, y_pred_test),
        "test_bal_acc": balanced_accuracy_score(y_test, y_pred_test),
        "test_f1_macro": f1_score(y_test, y_pred_test, average="macro"),
        "test_logloss": log_loss(y_test, y_proba_test, labels=labels),
    })

df = pd.DataFrame(rows).set_index("K")

# nicer formatting for visualization
df = df.round({
    "train_acc": 3,
    "train_bal_acc": 3,
    "train_f1_macro": 3,
    "train_logloss": 6,
    "test_acc": 3,
    "test_bal_acc": 3,
    "test_f1_macro": 3,
    "test_logloss": 6,
})

df

,train_acc,train_bal_acc,train_f1_macro,train_logloss,test_acc,test_bal_acc,test_f1_macro,test_logloss
K,,,,,,,,
1,1.0,1.0,1.0,0.000000,0.912,0.912,0.902,3.153820
2,1.0,1.0,1.0,0.000000,0.912,0.912,0.902,2.344260
3,1.0,1.0,1.0,0.000000,0.888,0.888,0.876,1.142885
4,1.0,1.0,1.0,0.000000,0.862,0.862,0.861,1.212248
5,1.0,1.0,1.0,0.000000,0.838,0.838,0.820,1.300020
10,1.0,1.0,1.0,0.000000,0.775,0.775,0.764,1.232000
12,1.0,1.0,1.0,0.000000,0.788,0.788,0.783,1.353588
15,1.0,1.0,1.0,0.000000,0.738,0.738,0.727,1.489528
20,1.0,1.0,1.0,0.000000,0.750,0.750,0.716,1.667509
